# Проект «Многоцелевая модель для NER и классификации»

## Краткое описание

Представьте, что вы NLP-инженер новостной аналитической платформы. Ваша система одновременно извлекает сущности из новостных текстов (NER, токен-уровень) и определяет набор прикладных событий и отношений в документе (CLS, документ-уровень). Это позволяет автоматически помечать важные факты в новостях (персоны, организации, даты) и формировать теги и инцидентные фильтры для быстрого поиска и аналитики.

Но есть проблема: текущий пайплайн раздельно выполняет NER и классификацию событий. В результате признаки дублируются, а вычислительные ресурсы используются неэффективно.

Вы предположили, что совместное обучение (multi-task) повысит качество NER и/или стабильность детекции событий. Чтобы проверить гипотезу, вам предстоит построить компактный воспроизводимый эксперимент, который:
- реализует объединённую encoder-модель, которая решает задачи NER и классификации событий;
- делает качественные и количественные выводы о взаимном влиянии задач.

## Данные

Используйте публичный датасет **NEREL** в формате JSONL: `train`, `dev`, `test`. Этот датасет содержит тексты, сущности (строки с офсетами) и отношения/события, его можно скачать на <a href="https://huggingface.co/datasets/iluvvatar/NEREL">huggingface</a>:

In [ ]:
wget -O train.jsonl "https://huggingface.co/datasets/iluvvatar/NEREL/resolve/main/data/train.jsonl"
wget -O dev.jsonl "https://huggingface.co/datasets/iluvvatar/NEREL/resolve/main/data/dev.jsonl"
wget -O test.jsonl "https://huggingface.co/datasets/iluvvatar/NEREL/resolve/main/data/test.jsonl"

wget -O ent_types.jsonl "https://huggingface.co/datasets/iluvvatar/NEREL/resolve/main/ent_types.jsonl"
wget -O rel_types.jsonl "https://huggingface.co/datasets/iluvvatar/NEREL/resolve/main/rel_types.jsonl"

## Инструкция по выполнению

### 1. Подготовка
- Используйте JSONL-файлы `train.jsonl`, `dev.jsonl`, `test.jsonl`.

### 2. EDA и обзор формата
- Загрузите несколько документов (train) и посмотрите примеры.
- Посчитайте частотности типов сущностей, а также типов их отношений.
- Постройте простые графики: топ‑15 типов сущностей, распределение длины текстов и сущностей на документ.
- Сделайте 2–3 вывода по EDA в Markdown.

### 3. Парсинг и подготовка таргетов
- Реализуйте и проверьте `parse_entity_line` и `parse_relation_line` (есть в шаблоне Jupyter).
- Напишите функцию `build_examples_from_nerel` — получите `tokens`, `token_spans`, `tags (BIO)`, `cls_vec (multihot)`.
- Постройте топ $K$ типов событий и отношений, $K≈30$.
- Сделайте sanity-check: для 5 случайных примеров выведите текст — токены — BIO — cls_vec.

### 4. Токенизация, выравнивание меток и DataLoader
- Выберите tokenizer (`AutoTokenizer(..., use_fast=True)`), реализуйте `tokenize_and_align_labels` (align word_ids → labels, pad/truncate).
- Соберите `Dataset`/`DataLoader` с `DataCollatorForTokenClassification` (или кастомный collator) для батчей.
- Проверьте shapes: `input_ids`, `labels` (с –100 для subword tokens), `cls_labels`.

### 5. Модель: JointModel + custom loss

- Реализуйте модель по шаблону encoder → dropout →
    - `token_cls` (линейный слой → токенный логит).
    - `cls_cls` (линейный слой от pooled `[CLS]` → multihot logits).
- Реализуйте `custom loss` для совместного обучения:
- Простая сумма: `loss = token_loss + cls_loss`.
- Альтернатива — **uncertainty weighting**: обучаемые `log_sigma_token`, `log_sigma_cls` и формула из Kendall et al.

    Задачи могут иметь разные масштабы потерь (CrossEntropy для токенов и BCE для multihot). Вместо ручного подбора весов для каждой задачи можно ввести параметры неопределённости (uncertainty). Для каждой задачи `i` вводят параметр `sigma_i > 0` (стандартное отклонение) и оптимизируют его вместе с сетью. Для задачи с loss L_i итоговая компонента loss принимается как

In [ ]:
(1 / (2 * sigma_i**2)) * L_i + log(sigma_i)

Для численной стабильности оптимизируют `log_sigma_i = log(sigma_i)`.

Тогда эквивалентная запись:

In [ ]:
exp(-2 * log_sigma_i) * L_i + log_sigma_i

**Плюсы:**
- не нужно подгонять веса вручную;
- оптимизация сама находит баланс между задачами;
- терм логарифма препятствует чрезмерному уменьшению sigma.

#### Реализация в PyTorch:

- В `JointModel.__init__` создайте параметры:

In [ ]:
self.log_sigma_token = nn.Parameter(torch.tensor(0.0))
self.log_sigma_cls = nn.Parameter(torch.tensor(0.0))

- Вычислите базовые losses:

In [ ]:
token_loss = token_loss_fct(token_logits.view(-1, C), labels.view(-1))
cls_loss = cls_loss_fct(cls_logits, cls_labels)

- Соберите итоговый loss по формуле:

In [ ]:
if self.use_uncertainty_weight:
    loss_token_term = torch.exp(-2.0 * self.log_sigma_token) * token_loss + self.log_sigma_token
    loss_cls_term = torch.exp(-2.0 * self.log_sigma_cls) * cls_loss + self.log_sigma_cls
    loss = loss_token_term + loss_cls_term
else:
    loss = token_loss + cls_loss

Так, если `log_sigma_token` убывает (sigma < 1), то вес для token_loss увеличивается.

Если `log_sigma_cls` растёт, то cls_loss получает меньший вклад.

### 6. Тренировка и валидация
- Настройте оптимизатор, LR-scheduler, gradient clipping.
- Обучите на достаточном количестве эпох.
- Рассчитайте качество на тестовой выборке:
    - token-level F1 (macro / seqeval) — используйте token-level mapping, либо преобразуйте предсказания в BIO и подсчитайте span-F1;
    - CLS micro-F1 (по flatten multihot), precision/recall.
- Соберите логи (таблица с epoch → loss → token_f1 → cls_f1).
- Сделайте выводы по проведённому обучению.

### 7. Инференс и качественный анализ
- Реализуйте функцию для предсказаний на инференсе: сырой текст → токены + predicted BIO + cls probabilities.
- На 8–10 ручных примерах оцените, где NER ошибается, как влияет CLS (ошибки, пропуски).
- Квантизируйте модель, сравните качество и скорость с базовой версией.